# Chapter 1.8: Sampling Strategies

## Learning Objectives

By the end of this notebook, you will:

1. **Implement** greedy decoding and understand its limitations
2. **Apply** temperature scaling with mathematical understanding
3. **Implement** top-k and top-p (nucleus) sampling from scratch
4. **Build** beam search and know when to use it
5. **Understand** repetition, frequency, and presence penalties
6. **Explore** min-p sampling as an alternative to top-p
7. **Get an overview** of speculative decoding

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.8_sampling.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.8_sampling.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional, Tuple
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)

# Create a simple vocabulary for demonstrations
VOCAB = ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'big', 'small', 'red',
         'blue', 'happy', 'sad', 'fast', 'slow', 'ate', 'fish', 'bird', 'tree', 'house',
         'is', 'was', 'very', 'and', 'but', 'or', 'a', 'an', 'in', 'with']
VOCAB_SIZE = len(VOCAB)
id_to_word = {i: w for i, w in enumerate(VOCAB)}
word_to_id = {w: i for i, w in enumerate(VOCAB)}

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Words: {VOCAB}")

---

## 1. From Logits to Tokens

A language model outputs **logits**: raw, unnormalized scores for each token in the vocabulary.

$$\text{logits} \in \mathbb{R}^{|V|}$$

To get probabilities, we apply **softmax**:

$$P(w_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$

The sampling strategy determines how we choose the next token from this distribution.

In [ ]:
## Figure: How Sampling Strategies Modify the Output Distribution
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np

VOCAB = ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'big', 'small', 'red',
         'blue', 'happy', 'sad', 'fast', 'slow', 'ate', 'fish', 'bird', 'tree', 'house']
VOCAB_SIZE = len(VOCAB)

# Create peaked logits
logits = torch.randn(VOCAB_SIZE) * 0.5
logits[1] = 5.0   # cat
logits[0] = 3.5   # the
logits[5] = 2.5   # dog
logits[2] = 2.0   # sat

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

BLUE, GREEN, ORANGE, RED = '#4A90D9', '#27AE60', '#F39C12', '#E74C3C'

def plot_dist(ax, probs, title, highlight_mask=None, color=BLUE):
    colors = [color if (highlight_mask is None or highlight_mask[i]) else '#BDC3C7' 
              for i in range(len(probs))]
    ax.bar(range(len(probs)), probs, color=colors, alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(VOCAB)))
    ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Probability')
    ax.grid(True, alpha=0.3, axis='y')

# Panel 1: Temperature
ax = axes[0, 0]
for T, color, label in [(0.3, RED, 'T=0.3 (focused)'), (1.0, BLUE, 'T=1.0 (default)'), (2.0, GREEN, 'T=2.0 (diverse)')]:
    probs = F.softmax(logits / T, dim=0).numpy()
    ax.plot(range(VOCAB_SIZE), probs, 'o-', color=color, linewidth=2, markersize=5, label=label, alpha=0.8)
ax.set_xticks(range(VOCAB_SIZE))
ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=8)
ax.set_title('Temperature Scaling', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Panel 2: Top-k
ax = axes[0, 1]
probs = F.softmax(logits, dim=0)
top_k = 5
_, top_idx = probs.topk(top_k)
mask = torch.zeros(VOCAB_SIZE, dtype=torch.bool)
mask[top_idx] = True
filtered = probs.clone()
filtered[~mask] = 0
filtered = filtered / filtered.sum()
plot_dist(ax, filtered.numpy(), f'Top-k Sampling (k={top_k})', mask.numpy(), GREEN)
ax.text(12, filtered.max().item() * 0.8, f'Only top-{top_k}\ntokens kept', fontsize=10,
        fontweight='bold', color=GREEN)

# Panel 3: Top-p
ax = axes[1, 0]
p_val = 0.9
sorted_probs, sorted_idx = probs.sort(descending=True)
cumsum = sorted_probs.cumsum(dim=0)
cutoff_mask = (cumsum - sorted_probs) <= p_val
n_kept = cutoff_mask.sum().item()
mask_orig = torch.zeros(VOCAB_SIZE, dtype=torch.bool)
mask_orig[sorted_idx[:n_kept]] = True
filtered_p = probs.clone()
filtered_p[~mask_orig] = 0
filtered_p = filtered_p / filtered_p.sum()
plot_dist(ax, filtered_p.numpy(), f'Top-p (Nucleus) Sampling (p={p_val})', mask_orig.numpy(), ORANGE)
ax.text(12, filtered_p.max().item() * 0.8, f'{n_kept} tokens kept\n(cumulative p >= {p_val})',
        fontsize=10, fontweight='bold', color=ORANGE)

# Panel 4: Min-p
ax = axes[1, 1]
min_p_val = 0.1
max_prob = probs.max()
threshold = min_p_val * max_prob
mask_minp = probs >= threshold
n_kept_minp = mask_minp.sum().item()
filtered_minp = probs.clone()
filtered_minp[~mask_minp] = 0
filtered_minp = filtered_minp / filtered_minp.sum()
plot_dist(ax, filtered_minp.numpy(), f'Min-p Sampling (min_p={min_p_val})', mask_minp.numpy(), RED)
ax.text(12, filtered_minp.max().item() * 0.8, f'{n_kept_minp} tokens kept\n(prob >= {min_p_val} * max)',
        fontsize=10, fontweight='bold', color=RED)
ax.axhline(y=threshold.item() / filtered_minp.sum().item(), color=RED, linestyle='--', alpha=0.5)

plt.suptitle('Sampling Strategies: How Each Modifies the Token Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("Temperature: controls sharpness. Top-k: fixed count. Top-p: adaptive by cumulative prob. Min-p: relative threshold.")

In [ ]:
def create_sample_logits(peaked: bool = True) -> torch.Tensor:
    """Create sample logits that look like a realistic model output."""
    logits = torch.randn(VOCAB_SIZE)
    if peaked:
        # Make 'cat' and 'the' much more likely
        logits[word_to_id['cat']] = 5.0
        logits[word_to_id['the']] = 3.5
        logits[word_to_id['dog']] = 2.5
        logits[word_to_id['sat']] = 2.0
    return logits


def visualize_distribution(logits: torch.Tensor, title: str = 'Token Distribution',
                          highlight_selected: Optional[int] = None, ax=None):
    """Visualize token probability distribution."""
    probs = F.softmax(logits, dim=0).numpy()
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 4))
    
    colors = ['#3498db'] * VOCAB_SIZE
    if highlight_selected is not None:
        colors[highlight_selected] = '#e74c3c'
    
    bars = ax.bar(range(VOCAB_SIZE), probs, color=colors, alpha=0.8, edgecolor='white')
    ax.set_xticks(range(VOCAB_SIZE))
    ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Probability')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='y')
    
    return ax


# Show the base distribution
logits = create_sample_logits(peaked=True)
probs = F.softmax(logits, dim=0)

fig, ax = plt.subplots(figsize=(14, 4))
visualize_distribution(logits, 'Raw Model Output (Logits -> Softmax)', ax=ax)
plt.tight_layout()
plt.show()

# Show top probabilities
top_probs, top_indices = probs.topk(5)
print("Top 5 tokens:")
for p, idx in zip(top_probs, top_indices):
    print(f"  {VOCAB[idx]:<10} P={p:.4f} ({p*100:.1f}%)")

---

## 2. Greedy Decoding

Simply pick the token with the highest probability:

$$w_t = \arg\max_{w} P(w | w_1, ..., w_{t-1})$$

**Pros**: Deterministic, fast

**Cons**: Repetitive, boring, can miss better overall sequences

In [ ]:
def greedy_decode(logits: torch.Tensor) -> int:
    """Select the token with the highest logit."""
    return logits.argmax().item()


# Demo
selected = greedy_decode(logits)
print(f"Greedy selection: '{VOCAB[selected]}' (id={selected})")

fig, ax = plt.subplots(figsize=(14, 4))
visualize_distribution(logits, f'Greedy Decoding: Always picks \'{VOCAB[selected]}\'', 
                       highlight_selected=selected, ax=ax)
plt.tight_layout()
plt.show()

# Show the problem with greedy: always picks the same token
print("\nGreedy is deterministic — running 10 times gives the same result:")
for i in range(10):
    selected = greedy_decode(logits)
    print(f"  Run {i}: '{VOCAB[selected]}'")

---

## 3. Temperature Scaling

Temperature $T$ controls the "sharpness" of the distribution:

$$P(w_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **T = 1.0**: Original distribution (default)
- **T < 1.0**: Sharper distribution (more confident, less diverse)
- **T > 1.0**: Flatter distribution (less confident, more diverse)
- **T -> 0**: Approaches greedy decoding
- **T -> inf**: Approaches uniform distribution

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """
    Apply temperature scaling to logits.
    
    temperature=1.0: no change
    temperature<1.0: sharper (more greedy)
    temperature>1.0: flatter (more random)
    """
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    return logits / temperature


# Visualize effect of temperature
temperatures = [0.1, 0.5, 1.0, 1.5, 2.0, 5.0]

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for ax, T in zip(axes, temperatures):
    scaled_logits = apply_temperature(logits, T)
    probs = F.softmax(scaled_logits, dim=0)
    entropy = -(probs * probs.log()).sum().item()
    
    visualize_distribution(scaled_logits, f'Temperature = {T} (entropy = {entropy:.2f})', ax=ax)

plt.suptitle('Effect of Temperature on Token Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Entropy analysis
temps = np.linspace(0.01, 5.0, 100)
entropies = []
top1_probs = []
for t in temps:
    probs = F.softmax(logits / t, dim=0)
    entropy = -(probs * probs.log()).sum().item()
    entropies.append(entropy)
    top1_probs.append(probs.max().item())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(temps, entropies, 'b-', linewidth=2)
axes[0].set_xlabel('Temperature', fontsize=12)
axes[0].set_ylabel('Entropy (nats)', fontsize=12)
axes[0].set_title('Entropy vs Temperature', fontsize=14)
axes[0].axhline(y=np.log(VOCAB_SIZE), color='red', linestyle='--', alpha=0.5, label=f'Max entropy (uniform)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(temps, top1_probs, 'r-', linewidth=2)
axes[1].set_xlabel('Temperature', fontsize=12)
axes[1].set_ylabel('Top-1 Probability', fontsize=12)
axes[1].set_title('Top-1 Token Probability vs Temperature', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. Top-k Sampling

Only consider the top $k$ tokens, setting all others to probability 0:

1. Sort tokens by probability
2. Keep only the top $k$ tokens
3. Re-normalize probabilities
4. Sample from the truncated distribution

In [ ]:
def top_k_sampling(logits: torch.Tensor, k: int, temperature: float = 1.0) -> int:
    """
    Top-k sampling: only consider the k most likely tokens.
    """
    # Apply temperature
    logits = logits / temperature
    
    # Get top-k
    top_k_logits, top_k_indices = logits.topk(k)
    
    # Convert to probabilities
    top_k_probs = F.softmax(top_k_logits, dim=0)
    
    # Sample
    selected_idx = torch.multinomial(top_k_probs, 1).item()
    
    return top_k_indices[selected_idx].item()


# Visualize top-k filtering
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, k in zip(axes, [3, 5, 10]):
    probs = F.softmax(logits, dim=0)
    top_k_vals, top_k_idx = probs.topk(k)
    
    # Create filtered distribution
    filtered = torch.zeros_like(probs)
    filtered[top_k_idx] = top_k_vals / top_k_vals.sum()  # renormalize
    
    colors = ['#e74c3c' if filtered[i] > 0 else '#bdc3c7' for i in range(VOCAB_SIZE)]
    ax.bar(range(VOCAB_SIZE), probs.numpy(), alpha=0.3, color='blue', label='Original')
    ax.bar(range(VOCAB_SIZE), filtered.numpy(), alpha=0.7, color=colors)
    ax.set_xticks(range(VOCAB_SIZE))
    ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Top-k = {k} (kept {top_k_vals.sum():.1%} of prob mass)', fontsize=13)
    ax.set_ylabel('Probability')

plt.suptitle('Top-k Sampling: Filtering to k Most Likely Tokens', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Sample multiple times
print("Top-5 sampling (10 samples):")
samples = [VOCAB[top_k_sampling(logits, k=5)] for _ in range(10)]
print(f"  Samples: {samples}")
print(f"  Distribution: {Counter(samples)}")

---

## 5. Top-p (Nucleus) Sampling

Instead of fixing $k$, keep the smallest set of tokens whose cumulative probability exceeds $p$:

1. Sort tokens by probability (descending)
2. Compute cumulative probability
3. Keep tokens until cumulative probability >= $p$
4. Re-normalize and sample

**Advantage over top-k**: Adapts to the shape of the distribution. When the model is confident, fewer tokens are kept. When uncertain, more tokens are included.

In [ ]:
def top_p_sampling(logits: torch.Tensor, p: float = 0.9, temperature: float = 1.0) -> int:
    """
    Top-p (nucleus) sampling: keep smallest set of tokens with cumulative prob >= p.
    """
    # Apply temperature
    logits = logits / temperature
    
    # Get probabilities
    probs = F.softmax(logits, dim=0)
    
    # Sort descending
    sorted_probs, sorted_indices = probs.sort(descending=True)
    
    # Cumulative sum
    cumulative_probs = sorted_probs.cumsum(dim=0)
    
    # Find cutoff: remove tokens with cumulative prob > p
    # Keep tokens where cumulative prob <= p (plus one more)
    mask = cumulative_probs - sorted_probs <= p  # shift by one to include the boundary token
    
    # Zero out filtered tokens
    filtered_probs = sorted_probs * mask.float()
    
    # Re-normalize
    filtered_probs = filtered_probs / filtered_probs.sum()
    
    # Sample
    selected_idx = torch.multinomial(filtered_probs, 1).item()
    
    return sorted_indices[selected_idx].item()


# Visualize top-p filtering
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax_row, logits_type in zip(axes, ['peaked', 'flat']):
    if logits_type == 'peaked':
        test_logits = create_sample_logits(peaked=True)
    else:
        test_logits = torch.randn(VOCAB_SIZE)  # flat distribution
    
    for ax, p_val in zip(ax_row, [0.5, 0.9, 0.95]):
        probs = F.softmax(test_logits, dim=0)
        sorted_probs, sorted_indices = probs.sort(descending=True)
        cumulative = sorted_probs.cumsum(dim=0)
        
        # Find how many tokens are kept
        mask = (cumulative - sorted_probs) <= p_val
        n_kept = mask.sum().item()
        
        # Show on original order
        kept_set = set(sorted_indices[:n_kept].tolist())
        colors = ['#2ecc71' if i in kept_set else '#bdc3c7' for i in range(VOCAB_SIZE)]
        
        ax.bar(range(VOCAB_SIZE), probs.numpy(), color=colors, alpha=0.8, edgecolor='white')
        ax.set_xticks(range(VOCAB_SIZE))
        ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=7)
        ax.set_title(f'Top-p={p_val} ({logits_type}): {n_kept} tokens kept', fontsize=11)
        ax.set_ylabel('Probability')

plt.suptitle('Top-p Sampling Adapts to Distribution Shape', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observation: With a peaked distribution, top-p=0.9 keeps fewer tokens.")
print("With a flat distribution, it keeps many more tokens.")
print("This adaptive behavior is why top-p is preferred over fixed top-k.")

---

## 6. Beam Search

Beam search maintains $B$ candidate sequences (beams) and expands each by all possible next tokens, keeping only the $B$ highest-scoring sequences.

Unlike greedy which only tracks one sequence, beam search explores multiple paths and finds a better (higher probability) complete sequence.

In [ ]:
def beam_search(
    get_logits_fn,  # function(token_ids) -> logits
    start_token: int,
    beam_width: int = 3,
    max_length: int = 5,
    eos_token: int = -1
) -> List[Tuple[List[int], float]]:
    """
    Beam search decoding.
    
    Returns: List of (token_ids, log_probability) tuples, sorted by probability.
    """
    # Each beam: (log_prob, token_ids)
    beams = [(0.0, [start_token])]
    completed = []
    
    for step in range(max_length):
        all_candidates = []
        
        for log_prob, token_ids in beams:
            # Get logits for next token
            logits = get_logits_fn(token_ids)
            log_probs = F.log_softmax(logits, dim=0)
            
            # Expand each beam with top-k candidates
            top_log_probs, top_indices = log_probs.topk(beam_width * 2)
            
            for lp, idx in zip(top_log_probs, top_indices):
                new_log_prob = log_prob + lp.item()
                new_ids = token_ids + [idx.item()]
                
                if idx.item() == eos_token:
                    completed.append((new_log_prob, new_ids))
                else:
                    all_candidates.append((new_log_prob, new_ids))
        
        # Keep top beam_width candidates
        all_candidates.sort(key=lambda x: x[0], reverse=True)
        beams = all_candidates[:beam_width]
        
        if not beams:
            break
    
    # Add remaining beams to completed
    completed.extend(beams)
    completed.sort(key=lambda x: x[0], reverse=True)
    
    return [(ids, lp) for lp, ids in completed]


# Demo with a simple mock language model
def mock_lm(token_ids):
    """Simple mock LM that returns logits based on last token."""
    last = token_ids[-1]
    logits = torch.randn(VOCAB_SIZE) * 0.5
    
    # Simple "grammar": some tokens are more likely after others
    if VOCAB[last] in ['the', 'a', 'an']:
        logits[word_to_id['cat']] += 3
        logits[word_to_id['dog']] += 2.5
        logits[word_to_id['big']] += 2
    elif VOCAB[last] in ['cat', 'dog']:
        logits[word_to_id['sat']] += 3
        logits[word_to_id['ran']] += 2.5
        logits[word_to_id['ate']] += 2
    elif VOCAB[last] in ['sat', 'ran']:
        logits[word_to_id['on']] += 3
        logits[word_to_id['in']] += 2
    elif VOCAB[last] in ['on', 'in']:
        logits[word_to_id['the']] += 3
        logits[word_to_id['a']] += 2
    
    return logits


# Run beam search
start = word_to_id['the']
results = beam_search(mock_lm, start, beam_width=3, max_length=5)

print("Beam Search Results (beam_width=3, max_length=5):")
print("=" * 60)
for i, (ids, log_prob) in enumerate(results[:5]):
    words = [VOCAB[idx] for idx in ids]
    print(f"  Beam {i}: {' '.join(words):<30} log_prob={log_prob:.3f}")

---

## 7. Repetition, Frequency, and Presence Penalties

These penalties reduce the likelihood of generating repeated tokens:

### Repetition Penalty
$$\text{logits}[i] = \begin{cases} \text{logits}[i] / \alpha & \text{if } \text{logits}[i] > 0 \text{ and } i \in \text{generated} \\ \text{logits}[i] \times \alpha & \text{if } \text{logits}[i] < 0 \text{ and } i \in \text{generated} \end{cases}$$

### Frequency Penalty (OpenAI-style)
$$\text{logits}[i] = \text{logits}[i] - \alpha_f \times \text{count}(i)$$

### Presence Penalty (OpenAI-style)
$$\text{logits}[i] = \text{logits}[i] - \alpha_p \times \mathbb{1}[i \in \text{generated}]$$

In [ ]:
def apply_repetition_penalty(
    logits: torch.Tensor,
    generated_tokens: List[int],
    penalty: float = 1.2
) -> torch.Tensor:
    """Apply repetition penalty to logits."""
    logits = logits.clone()
    for token_id in set(generated_tokens):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits


def apply_frequency_penalty(
    logits: torch.Tensor,
    generated_tokens: List[int],
    penalty: float = 0.5
) -> torch.Tensor:
    """Apply frequency penalty (proportional to count)."""
    logits = logits.clone()
    counts = Counter(generated_tokens)
    for token_id, count in counts.items():
        logits[token_id] -= penalty * count
    return logits


def apply_presence_penalty(
    logits: torch.Tensor,
    generated_tokens: List[int],
    penalty: float = 0.5
) -> torch.Tensor:
    """Apply presence penalty (binary: present or not)."""
    logits = logits.clone()
    for token_id in set(generated_tokens):
        logits[token_id] -= penalty
    return logits


# Visualize penalty effects
generated = [word_to_id['cat'], word_to_id['cat'], word_to_id['the'], word_to_id['sat']]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Original
visualize_distribution(logits, 'Original Distribution', ax=axes[0, 0])

# Repetition penalty
logits_rep = apply_repetition_penalty(logits, generated, penalty=1.5)
visualize_distribution(logits_rep, 'Repetition Penalty (1.5)', ax=axes[0, 1])

# Frequency penalty
logits_freq = apply_frequency_penalty(logits, generated, penalty=1.0)
visualize_distribution(logits_freq, 'Frequency Penalty (1.0)', ax=axes[1, 0])

# Presence penalty
logits_pres = apply_presence_penalty(logits, generated, penalty=1.0)
visualize_distribution(logits_pres, 'Presence Penalty (1.0)', ax=axes[1, 1])

# Mark previously generated tokens
for ax in axes.flatten():
    for token_id in set(generated):
        ax.axvline(x=token_id, color='red', linestyle=':', alpha=0.3)

plt.suptitle(f'Penalty Effects (previously generated: {[VOCAB[t] for t in generated]})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 8. Min-p Sampling

Min-p (proposed by FALD) is an alternative to top-p that sets a dynamic probability floor:

$$\text{threshold} = p_{\text{min}} \times \max(P)$$

Keep all tokens with probability >= threshold.

**Advantage**: More intuitive parameter. min_p=0.1 means "keep any token with at least 10% of the top token's probability."

In [ ]:
def min_p_sampling(logits: torch.Tensor, min_p: float = 0.1, temperature: float = 1.0) -> int:
    """
    Min-p sampling: keep tokens with prob >= min_p * max_prob.
    """
    logits = logits / temperature
    probs = F.softmax(logits, dim=0)
    
    # Dynamic threshold
    max_prob = probs.max()
    threshold = min_p * max_prob
    
    # Filter
    mask = probs >= threshold
    filtered_probs = probs * mask.float()
    
    # Re-normalize
    filtered_probs = filtered_probs / filtered_probs.sum()
    
    # Sample
    return torch.multinomial(filtered_probs, 1).item()


# Compare min-p vs top-p
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Top-p
for ax, p_val in zip(axes[0], [0.5, 0.9, 0.95]):
    probs = F.softmax(logits, dim=0)
    sorted_probs, sorted_indices = probs.sort(descending=True)
    cumulative = sorted_probs.cumsum(dim=0)
    mask = (cumulative - sorted_probs) <= p_val
    n_kept = mask.sum().item()
    kept_set = set(sorted_indices[:n_kept].tolist())
    colors = ['#2ecc71' if i in kept_set else '#bdc3c7' for i in range(VOCAB_SIZE)]
    ax.bar(range(VOCAB_SIZE), probs.numpy(), color=colors, alpha=0.8)
    ax.set_xticks(range(VOCAB_SIZE))
    ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Top-p={p_val}: {n_kept} tokens', fontsize=12)
    ax.set_ylabel('Probability')

# Min-p
for ax, min_p_val in zip(axes[1], [0.05, 0.1, 0.2]):
    probs = F.softmax(logits, dim=0)
    threshold = min_p_val * probs.max()
    mask = probs >= threshold
    n_kept = mask.sum().item()
    colors = ['#e74c3c' if mask[i] else '#bdc3c7' for i in range(VOCAB_SIZE)]
    ax.bar(range(VOCAB_SIZE), probs.numpy(), color=colors, alpha=0.8)
    ax.set_xticks(range(VOCAB_SIZE))
    ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Min-p={min_p_val}: {n_kept} tokens (threshold={threshold:.4f})', fontsize=12)
    ax.set_ylabel('Probability')

axes[0][0].set_ylabel('Top-p\nProbability', fontsize=12)
axes[1][0].set_ylabel('Min-p\nProbability', fontsize=12)

plt.suptitle('Top-p vs Min-p Sampling', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 9. Speculative Decoding Overview

**Problem**: Decode is memory-bound, and GPU is underutilized. Can we decode faster?

**Speculative Decoding Idea**:
1. Use a small, fast **draft model** to generate $K$ candidate tokens quickly
2. **Verify** all $K$ tokens in parallel using the large target model (one forward pass)
3. **Accept** tokens that the target model agrees with, **reject** and resample where it disagrees

**Key property**: The output distribution is **exactly the same** as the target model. Speculative decoding is a speedup technique, not an approximation!

$$\text{Speedup} \approx \frac{K \times \alpha}{1 + K \times \text{cost ratio}}$$

where $\alpha$ is the average acceptance rate.

In [ ]:
def simulate_speculative_decoding(
    n_tokens: int = 100,
    K: int = 5,              # number of speculative tokens
    accept_rate: float = 0.7, # average acceptance rate
    target_time_ms: float = 30.0,  # target model forward pass time
    draft_time_ms: float = 3.0,    # draft model forward pass time
    target_verify_time_ms: float = 35.0,  # target model verify K tokens time
):
    """
    Simulate speculative decoding to estimate speedup.
    """
    # Standard autoregressive: one token at a time
    standard_time = n_tokens * target_time_ms
    
    # Speculative: draft K tokens, verify, accept some
    tokens_generated = 0
    spec_time = 0
    n_rounds = 0
    accepted_per_round = []
    
    while tokens_generated < n_tokens:
        # Draft K tokens
        spec_time += K * draft_time_ms
        
        # Verify all K tokens in one target forward pass
        spec_time += target_verify_time_ms
        
        # Accept tokens (simplified: geometric distribution)
        accepted = 0
        for i in range(K):
            if np.random.random() < accept_rate:
                accepted += 1
            else:
                break
        # Always get at least 1 token (resample the rejected one)
        accepted = max(1, accepted)
        
        tokens_generated += accepted
        accepted_per_round.append(accepted)
        n_rounds += 1
    
    return {
        'standard_time_ms': standard_time,
        'speculative_time_ms': spec_time,
        'speedup': standard_time / spec_time,
        'avg_accepted': np.mean(accepted_per_round),
        'n_rounds': n_rounds,
        'tokens_per_round': n_tokens / n_rounds,
    }


# Simulate with different acceptance rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

accept_rates = np.linspace(0.3, 0.95, 20)
K_values = [3, 5, 7, 10]

for K in K_values:
    speedups = []
    for ar in accept_rates:
        result = simulate_speculative_decoding(n_tokens=200, K=K, accept_rate=ar)
        speedups.append(result['speedup'])
    axes[0].plot(accept_rates * 100, speedups, linewidth=2, label=f'K={K}')

axes[0].set_xlabel('Acceptance Rate (%)', fontsize=12)
axes[0].set_ylabel('Speedup', fontsize=12)
axes[0].set_title('Speculative Decoding Speedup', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='No speedup')

# Tokens per round
for K in K_values:
    tpr = []
    for ar in accept_rates:
        result = simulate_speculative_decoding(n_tokens=200, K=K, accept_rate=ar)
        tpr.append(result['tokens_per_round'])
    axes[1].plot(accept_rates * 100, tpr, linewidth=2, label=f'K={K}')

axes[1].set_xlabel('Acceptance Rate (%)', fontsize=12)
axes[1].set_ylabel('Tokens per Verification Round', fontsize=12)
axes[1].set_title('Effective Tokens per Round', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show one specific result
result = simulate_speculative_decoding(n_tokens=200, K=5, accept_rate=0.7)
print(f"\nSpeculative Decoding (K=5, accept_rate=70%):")
print(f"  Standard time:    {result['standard_time_ms']:.0f} ms")
print(f"  Speculative time: {result['speculative_time_ms']:.0f} ms")
print(f"  Speedup:          {result['speedup']:.2f}x")
print(f"  Avg accepted:     {result['avg_accepted']:.1f} tokens/round")
print(f"  Verification rounds: {result['n_rounds']}")

---

## 10. Interactive Sampling Comparison

Let's compare all sampling methods side-by-side.

In [ ]:
def compare_sampling_methods(logits: torch.Tensor, n_samples: int = 1000):
    """
    Compare different sampling methods by drawing many samples.
    """
    methods = {
        'Greedy': lambda: greedy_decode(logits),
        'Temp=0.5': lambda: torch.multinomial(F.softmax(logits/0.5, dim=0), 1).item(),
        'Temp=1.0': lambda: torch.multinomial(F.softmax(logits, dim=0), 1).item(),
        'Temp=1.5': lambda: torch.multinomial(F.softmax(logits/1.5, dim=0), 1).item(),
        'Top-k=3': lambda: top_k_sampling(logits, k=3),
        'Top-k=10': lambda: top_k_sampling(logits, k=10),
        'Top-p=0.5': lambda: top_p_sampling(logits, p=0.5),
        'Top-p=0.9': lambda: top_p_sampling(logits, p=0.9),
        'Min-p=0.1': lambda: min_p_sampling(logits, min_p=0.1),
    }
    
    results = {}
    for name, sample_fn in methods.items():
        samples = [sample_fn() for _ in range(n_samples)]
        counter = Counter(samples)
        results[name] = counter
    
    return results


results = compare_sampling_methods(logits, n_samples=2000)

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for ax, (name, counter) in zip(axes, results.items()):
    counts = [counter.get(i, 0) for i in range(VOCAB_SIZE)]
    total = sum(counts)
    freqs = [c / total for c in counts]
    
    # Sort by frequency
    sorted_indices = sorted(range(VOCAB_SIZE), key=lambda i: freqs[i], reverse=True)
    
    colors = plt.cm.Set2(np.linspace(0, 1, VOCAB_SIZE))
    ax.bar(range(VOCAB_SIZE), [freqs[i] for i in sorted_indices], 
           color=[colors[i] for i in sorted_indices], alpha=0.8)
    ax.set_xticks(range(min(15, VOCAB_SIZE)))
    ax.set_xticklabels([VOCAB[sorted_indices[i]] for i in range(min(15, VOCAB_SIZE))], 
                       rotation=45, ha='right', fontsize=8)
    
    unique = len([c for c in counts if c > 0])
    ax.set_title(f'{name} ({unique} unique tokens)', fontsize=12)
    ax.set_ylabel('Frequency')

plt.suptitle('Sampling Method Comparison (2000 samples each)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Exercises

### Exercise 1: Implement All Sampling Methods from Scratch

Implement a unified sampling function that supports all methods.

In [ ]:
def unified_sample(
    logits: torch.Tensor,
    temperature: float = 1.0,
    top_k: int = 0,       # 0 = disabled
    top_p: float = 1.0,   # 1.0 = disabled
    min_p: float = 0.0,   # 0.0 = disabled
    repetition_penalty: float = 1.0,  # 1.0 = disabled
    frequency_penalty: float = 0.0,   # 0.0 = disabled
    presence_penalty: float = 0.0,    # 0.0 = disabled
    generated_tokens: List[int] = None,
) -> int:
    """
    Exercise: Implement a unified sampling function.
    
    Apply transformations in this order:
    1. Repetition/frequency/presence penalties
    2. Temperature scaling
    3. Top-k filtering
    4. Top-p filtering
    5. Min-p filtering
    6. Sample from filtered distribution
    """
    # YOUR CODE HERE
    pass

# Test:
# token = unified_sample(logits, temperature=0.8, top_p=0.9, repetition_penalty=1.2,
#                        generated_tokens=[1, 1, 3])
# print(f"Sampled: {VOCAB[token]}")

### Exercise 2: Sampling Quality Analysis

Generate sequences with different sampling configurations and analyze:
1. Token diversity (unique tokens / total tokens)
2. Repetition rate (consecutive same token)
3. Perplexity under the model distribution

In [ ]:
def analyze_sampling_quality(
    get_logits_fn,
    sample_fn,
    n_tokens: int = 100,
    start_token: int = 0
):
    """
    Exercise: Generate a sequence and compute quality metrics.
    
    Args:
        get_logits_fn: function(token_ids) -> logits tensor
        sample_fn: function(logits) -> int (sampled token id)
        n_tokens: number of tokens to generate
        start_token: initial token id to start the sequence
    
    Returns a dict with these metrics:
        - "sequence": List[int] — the generated token ids
        - "diversity": float — len(unique_tokens) / n_tokens
        - "repetition_rate": float — (count of i where token[i]==token[i-1]) / (n_tokens - 1)
        - "avg_log_prob": float — mean of log P(token_i | context) across all generated tokens
        - "top1_agreement": float — fraction of tokens that match the greedy (argmax) choice
    
    Steps:
    1. TODO: Initialize the sequence with [start_token]
    2. TODO: Loop n_tokens times:
       a. Call get_logits_fn(sequence_so_far) to get logits
       b. Call sample_fn(logits) to pick the next token
       c. Compute log_softmax(logits) and record the log-prob of the chosen token
       d. Check if the chosen token equals argmax(logits) for top1_agreement
       e. Append the token to the sequence
    3. TODO: Compute diversity = len(set(generated)) / n_tokens
    4. TODO: Compute repetition_rate = number of consecutive duplicates / (n_tokens - 1)
    5. TODO: Compute avg_log_prob = mean of all recorded log-probs
    6. TODO: Compute top1_agreement = count of argmax matches / n_tokens
    7. TODO: Return the metrics dict
    """
    # TODO: Implement the generation loop and metric computation
    pass

# Test:
# greedy_metrics = analyze_sampling_quality(mock_lm, lambda l: greedy_decode(l), n_tokens=50)
# print("Greedy decoding metrics:")
# for k, v in greedy_metrics.items():
#     if k != "sequence":
#         print(f"  {k}: {v:.4f}")
#
# topk_metrics = analyze_sampling_quality(mock_lm, lambda l: top_k_sampling(l, k=5), n_tokens=50)
# print("\nTop-k=5 sampling metrics:")
# for k, v in topk_metrics.items():
#     if k != "sequence":
#         print(f"  {k}: {v:.4f}")

---

## Summary

### Key Takeaways

1. **Greedy decoding** always picks the most likely token. Deterministic but repetitive.

2. **Temperature** controls distribution sharpness. T<1 = more focused, T>1 = more random. It's the most basic sampling control.

3. **Top-k** keeps only the k most likely tokens. Simple but uses a fixed k regardless of distribution shape.

4. **Top-p (nucleus)** dynamically adjusts the number of candidates based on cumulative probability. Adapts to model confidence.

5. **Min-p** sets a relative threshold based on the top token's probability. More intuitive than top-p in some cases.

6. **Beam search** explores multiple hypotheses in parallel. Best for tasks with a single correct answer (translation, summarization).

7. **Penalties** (repetition, frequency, presence) reduce repetition by modifying logits of previously generated tokens.

8. **Speculative decoding** uses a draft model to generate candidates, verified in parallel by the target model. Exact same output distribution with significant speedup.

### Practical Recommendations

| Use Case | Recommended Settings |
|----------|---------------------|
| Code generation | temp=0.2, top_p=0.95 |
| Creative writing | temp=0.9, top_p=0.95, freq_penalty=0.3 |
| Chatbot | temp=0.7, top_p=0.9 |
| Translation | beam search (width=5) or temp=0.3 |
| Factual Q&A | temp=0.1 or greedy |

### What's Next

This completes **Part 1: Foundations of LLM Serving**. In **Part 2**, we'll dive into the vLLM and SGLang codebases to see how these foundational concepts are implemented in production-grade serving systems.